### results

everything here reads from `runs/`. nothing runs a GPU.

In [ ]:
import json
from pathlib import Path

FT = Path.cwd().parent
RUNS = FT / "runs"

def pooled(tag, model, arm):
    p = RUNS / tag / f"{model}-{arm}" / "summary.json"
    if not p.exists():
        return None
    s = json.loads(p.read_text())
    n = sum(d["n"] for d in s.values())
    return 100 * sum(d["correct"] for d in s.values()) / n

pooled("ft-v2", "8b", "story")

### the main table

base vs fine-tuned, three models, three grammars x two input arms.
A = trained grammar, b-near = same shape different words, b-far = different structure.

In [ ]:
BASE_A = {"8b": "base-v1", "qwen3-32b": "base-v1", "ministral-14b": "base-v2"}
MODELS = [("8b", "8B"), ("ministral-14b", "14B"), ("qwen3-32b", "32B")]
ARMS = ["story", "literal", "story-bnear", "literal-bnear", "story-bfar", "literal-bfar"]

print(f"{'arm':16s}" + "".join(f"{lbl:>16s}" for _, lbl in MODELS))
for arm in ARMS:
    cells = []
    for key, _ in MODELS:
        tag = BASE_A[key] if "-" not in arm else "base-v2"
        b, f = pooled(tag, key, arm), pooled("ft-v2", key, arm)
        cells.append(f"{b:.1f} -> {f:.1f}" if b is not None and f is not None else "-")
    print(f"{arm:16s}" + "".join(f"{c:>16s}" for c in cells))

### per tier

pooled numbers hide the depth story — order5 is the deepest tier.

In [ ]:
s = json.loads((RUNS / "ft-v2/qwen3-32b-story/summary.json").read_text())
for tier, d in sorted(s.items()):
    print(f"{tier:12s} n={d['n']:3d}  correct {d['correct_pct']:5.1f}%  unparseable {d['unparseable_pct']:5.1f}%")

### what the model actually wrote

the numbers are only as good as the outputs behind them. read some.

In [ ]:
rows = [json.loads(l) for l in open(RUNS / "ft-v2/8b-story/results.jsonl")]
r = rows[3]
print(r["verdict"]["status"], "|", r["bucket"])
print(r["response"][-200:])

In [ ]:
# the far-grammar failures are the interesting ones
bad = [r for r in (json.loads(l) for l in open(RUNS / "ft-v2/ministral-14b-story-bfar/results.jsonl"))
       if r["verdict"]["status"] == "unparseable"]
print(len(bad), "unparseable")
for r in bad[:3]:
    line = [l for l in r["response"].splitlines() if l.startswith("DERIVE")]
    if line:
        print(f"parens {line[0].count('(')}/{line[0].count(')')} |", line[0][:100])

### checkpoint curves

trained grammar vs never-trained grammar, over training. this is where the
"transfer peaks early then decays" finding comes from.

In [ ]:
def curve(run, model, arm):
    out = []
    for d in sorted(RUNS.glob(f"curve-v2-{run}-step-*")):
        step = int(d.name.rsplit("step-", 1)[1])
        p = d / f"{model}-{arm}-limit200" / "summary.json"
        if p.exists():
            s = json.loads(p.read_text())
            n = sum(x["n"] for x in s.values())
            out.append((step, 100 * sum(x["correct"] for x in s.values()) / n))
    return sorted(out)

curve("v2-ministral-14b-s0", "ministral-14b", "story-bfar")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, (run, model, label) in zip(axes, [
        ("v2-8b-s0", "8b", "8B"),
        ("v2-ministral-14b-s0", "ministral-14b", "14B"),
        ("v2-qwen3-32b-s0", "qwen3-32b", "32B")]):
    for arm, lab in (("story", "grammar A"), ("story-bfar", "b-far")):
        pts = curve(run, model, arm)
        ax.plot([s for s, _ in pts], [v for _, v in pts], "o-", label=lab)
    ax.set_title(label); ax.set_xlabel("step"); ax.grid(alpha=.3)
axes[0].set_ylabel("correct %"); axes[0].legend()
plt.tight_layout(); plt.show()

### figures for the writeup

`python3 make_figures.py` writes both into `assets/`.

In [ ]:
!python3 make_figures.py